# AMPS II · Week 13 — HW12
## Brinkman layers, the porous interface, Forchheimer and Ergun, conjugate heat transfer, contact resistance and an LLM-generated heat-transfer script

**Due before Week 14 · 100 points.** Submit the executed notebook: *Runtime → Restart session and run all*, fill in the AI-usage log at the end.
Every number you report must come from a cell of this notebook. The first cell of Q7 runs an LLM-generated script that is **meant** to be wrong. The whole HW runs in a few minutes.

| Q | Topic | Points |
|---|---|:---:|
| 1 | Theory: Brinkman, the one-domain interface, the developed temperature, the consistent flux | 15 |
| 2 | Resolving the Brinkman layer: uniform and graded meshes | 14 |
| 3 | The porous interface: slip, shear and the Beavers–Joseph coefficient | 14 |
| 4 | Forchheimer: the Ergun pressure drop of a packed bed | 13 |
| 5 | Conjugate heat transfer: the wall conductivity | 15 |
| 6 | Contact resistance and the conditioning of a penalty | 14 |
| 7 | Debugging an LLM-generated heat-transfer script | 15 |

## ⚙️ Setup — run once per runtime
Installs DOLFINx 0.11 and registers the `%%fenicsx` cell magic. Re-run after every runtime restart.

In [ ]:
# ══════════════════════════════════════════════════════════════════
#  FEniCSx on Google Colab — run once per runtime (≈ 3–5 min)
#  https://github.com/SeoulTechPSE/fenicsx-colab
# ══════════════════════════════════════════════════════════════════
from google.colab import drive
import os, multiprocessing, subprocess
from pathlib import Path

if not os.path.ismount('/content/drive'):
    drive.mount('/content/drive')

REPO_URL = 'https://github.com/seoultechpse/fenicsx-colab.git'
REPO_DIR = Path('/content/fenicsx-colab')
if not REPO_DIR.exists():
    subprocess.run(['git', 'clone', REPO_URL, str(REPO_DIR)], check=True)

# OpenMPI 5.x slot patch: Colab has 2 CPUs; allow up to 4 ranks (oversubscribed)
N_PROC = 4
with open('/root/hostfile', 'w') as f:
    f.write(f'localhost slots={N_PROC}\n')
os.environ['OMPI_MCA_rmaps_default_mapping_policy'] = 'slot:OVERSUBSCRIBE'
os.environ['PRTE_MCA_rmaps_default_mapping_policy'] = 'slot:OVERSUBSCRIBE'
os.environ['OMPI_MCA_osc'] = '^ucx'   # skip the UCX one-sided module (harmless start-up message)

DOLFINX_VERSION = '0.11'
USE_COMPLEX = False
USE_CLEAN   = False
opts = [f'--version {DOLFINX_VERSION}']
if USE_COMPLEX: opts.append('--complex')
if USE_CLEAN:   opts.append('--clean')
get_ipython().run_line_magic('run', f"{REPO_DIR / 'setup_fenicsx.py'} {' '.join(opts)}")

# Folder shared by %%fenicsx cells (computation) and ordinary cells (plotting)
os.makedirs('/tmp/fnx', exist_ok=True)

In [ ]:
%%fenicsx --info -np 2

The model module from the lab (unchanged). Run the cell once.

In [ ]:
%%writefile /tmp/fnx/porous.py
"""porous.py — porous media (Brinkman, Brinkman-Forchheimer) and conjugate heat transfer on submeshes, DOLFINx 0.11 (AMPS II, Week 13)."""
import gc
import numpy as np
import ufl, basix.ufl
from mpi4py import MPI
from dolfinx import fem, mesh, geometry
from dolfinx.fem.petsc import NonlinearProblem, LinearProblem

comm = MPI.COMM_WORLD
gc.disable()                                   # PETSc destructors are collective: collect at the same point on all ranks
def collect():
    gc.collect()
def say(*a):
    if comm.rank == 0:
        print(*a, flush=True)
LU = {"ksp_type": "preonly", "pc_type": "lu", "pc_factor_mat_solver_type": "mumps"}
NEWTON = dict(LU, snes_type="newtonls", snes_linesearch_type="none", snes_rtol=1e-10, snes_atol=1e-11, snes_stol=0.0, snes_max_it=30)

def channel(L=4.0, y0=0.0, y1=1.0, nx=8, ny=16):
    return mesh.create_rectangle(comm, [[0, y0], [L, y1]], [nx, ny], mesh.CellType.quadrilateral)

def tag_cells(msh, a, lower=2, upper=1):
    """cell tags by the cell midpoint: y > a -> upper (1), y < a -> lower (2)"""
    tdim = msh.topology.dim; n = msh.topology.index_map(tdim).size_local + msh.topology.index_map(tdim).num_ghosts
    mid = mesh.compute_midpoints(msh, tdim, np.arange(n, dtype=np.int32))
    return mesh.meshtags(msh, tdim, np.arange(n, dtype=np.int32), np.where(mid[:, 1] > a, upper, lower).astype(np.int32))

def probe(msh, f, P, comp=0):
    tdim = msh.topology.dim; nloc = msh.topology.index_map(tdim).size_local
    cand = geometry.compute_colliding_cells(msh, geometry.compute_collisions_points(geometry.bb_tree(msh, tdim), P), P)
    out, cnt = np.zeros(len(P)), np.zeros(len(P)); idx, cells = [], []
    for j in range(len(P)):
        own = [c for c in cand.links(j) if c < nloc]
        if own:
            idx.append(j); cells.append(own[0])
    if idx:
        val = np.asarray(f.eval(P[idx], np.array(cells, dtype=np.int32))).reshape(len(idx), -1)
        out[idx] = val[:, comp]; cnt[idx] = 1
    return comm.allreduce(out, op=MPI.SUM) / np.maximum(comm.allreduce(cnt, op=MPI.SUM), 1)

def column(msh, f, x, y0, y1, n=201, comp=0):
    P = np.zeros((n, 3)); P[:, 0] = x; P[:, 1] = np.linspace(y0, y1, n)
    return P[:, 1], probe(msh, f, P, comp)

# ═══════════════════════════════════════════════ Part 1: Brinkman / Brinkman-Forchheimer
class Brinkman:
    """-mu_e lap u + (mu/K) u + beta |u| u + grad p = G e_x, div u = 0 on a channel; walls y = y0, y1 no slip; ends do-nothing (p = 0)
    invK: a number or a DG0 Function (1/K, zero in free fluid); beta: Forchheimer coefficient (number or DG0)."""
    count = 0
    def __init__(self, msh, invK=0.0, G=1.0, mue=1.0, mu=1.0, beta=0.0):
        collect(); Brinkman.count += 1
        self.msh = msh; c = msh.basix_cell()
        W = self.W = fem.functionspace(msh, basix.ufl.mixed_element([basix.ufl.element("Lagrange", c, 2, shape=(2,)), basix.ufl.element("Lagrange", c, 1)]))
        self.w = fem.Function(W); u, p = ufl.split(self.w); v, q = ufl.TestFunctions(W)
        un = ufl.sqrt(ufl.dot(u, u) + 1e-20)
        F = (mue * ufl.inner(ufl.grad(u), ufl.grad(v)) + mu * invK * ufl.dot(u, v) + beta * un * ufl.dot(u, v)
             - p * ufl.div(v) - q * ufl.div(u) - G * v[0]) * ufl.dx
        x = msh.geometry.x; y0 = comm.allreduce(x[:, 1].min(), op=MPI.MIN); y1 = comm.allreduce(x[:, 1].max(), op=MPI.MAX)
        V, _ = W.sub(0).collapse()
        walls = mesh.locate_entities_boundary(msh, 1, lambda x: np.isclose(x[1], y0) | np.isclose(x[1], y1))
        self.bcs = [fem.dirichletbc(fem.Function(V), fem.locate_dofs_topological((W.sub(0), V), 1, walls), W.sub(0))]
        self.pb = NonlinearProblem(F, self.w, bcs=self.bcs, petsc_options_prefix=f"bk{Brinkman.count}_", petsc_options=NEWTON)
        self.y0, self.y1 = y0, y1; self.L = comm.allreduce(x[:, 0].max(), op=MPI.MAX)

    def solve(self):
        self.pb.solve(); s = self.pb.solver
        return int(s.getConvergedReason()), int(s.getIterationNumber())

    def u(self):
        return self.w.sub(0).collapse()

    def flow_rate(self, x=None):
        """volume flux through the section x (default: the middle) per unit depth"""
        y, ux = column(self.msh, self.u(), self.L / 2 if x is None else x, self.y0, self.y1, n=2001)
        return np.trapezoid(ux, y)

def invK_field(msh, ct, K, free_tag=1, porous_tag=2):
    """DG0 1/K: 0 in the free fluid, 1/K in the porous layer"""
    Q = fem.functionspace(msh, ("DG", 0)); f = fem.Function(Q)
    f.x.array[:] = 0.0; f.x.array[ct.find(porous_tag)] = 1.0 / K
    return f

def brinkman_exact(y, K, G=1.0, mu=1.0, mue=1.0):
    """porous channel 0 < y < 1 (walls no slip): u = G K/mu (1 - cosh((y - 1/2)/lam) / cosh(1/(2 lam))), lam = sqrt(mue K/mu)"""
    lam = np.sqrt(mue * K / mu)
    return G * K / mu * (1 - np.cosh((y - 0.5) / lam) / np.cosh(0.5 / lam))

def two_layer_exact(y, a, K, G=1.0, mu=1.0, mue=1.0):
    """porous layer 0 < y < a (Brinkman), free fluid a < y < 1 (Stokes), no slip at y = 0, 1; u and the shear stress continuous at y = a.
    Porous: u = ud + A exp(-y/lam) + B exp((y - a)/lam) (well scaled for thin layers); free: u = -G y^2/(2 mu) + C y + D."""
    lam = np.sqrt(mue * K / mu); ud = G * K / mu; e = np.exp(-a / lam)
    M = np.array([[1.0, e, 0.0, 0.0],                                        # u(0) = 0
                  [0.0, 0.0, 1.0, 1.0],                                      # u(1) = 0
                  [e, 1.0, -a, -1.0],                                        # continuity of u at a
                  [-mue / lam * e, mue / lam, -mu, 0.0]])                    # continuity of the shear stress at a
    rhs = np.array([-ud, G / (2 * mu), -ud - G / (2 * mu) * a ** 2, -G * a])
    A, B, C, D = np.linalg.solve(M, rhs)
    y = np.asarray(y, dtype=float); yp = np.minimum(y, a)
    return np.where(y < a, ud + A * np.exp(-yp / lam) + B * np.exp((yp - a) / lam), -G / (2 * mu) * y ** 2 + C * y + D)

def ergun(G, invK, beta, mu=1.0):
    """core (Darcy-Forchheimer) velocity: G = mu U / K + beta U^2"""
    if beta == 0:
        return G / (mu * invK)
    return (-mu * invK + np.sqrt((mu * invK) ** 2 + 4 * beta * G)) / (2 * beta)

# ═══════════════════════════════════════════════ Part 2: conjugate heat transfer
class Conjugate:
    """fluid channel 0 < y < 1 (tag 1) on a solid wall -t < y < 0 (tag 2), length L; heat flux q into the solid at y = -t, top y = 1 adiabatic.
    Flow: Stokes on the FLUID SUBMESH (parabolic inlet, do-nothing outlet), transferred to the parent mesh (zero in the solid).
    Temperature (nondimensional with H, mean velocity, fluid conductivity): Pe u . grad T = div(k grad T), k = 1 (fluid), ks (solid).
    mode "single": one P2 space on the parent mesh (T continuous); mode "contact": one space per submesh, coupled on the interface by a
    contact conductance hc: flux = hc (T_s - T_f) (hc -> infinity: perfect contact)."""
    count = 0
    def __init__(self, msh, ct, ks=10.0, Pe=20.0, q=1.0, mode="single", hc=None, flow="stokes", deg=2, rtol=1e-10):
        collect(); Conjugate.count += 1; cnt = Conjugate.count
        self.msh, self.ct = msh, ct; tdim = msh.topology.dim; fdim = tdim - 1
        x = msh.geometry.x; self.L = comm.allreduce(x[:, 0].max(), op=MPI.MAX); self.t = -comm.allreduce(x[:, 1].min(), op=MPI.MIN)
        self.Pe, self.ks, self.q = Pe, ks, q
        # ── flow on the fluid submesh ──
        self.fsub, self.fem_, _, _ = mesh.create_submesh(msh, tdim, ct.find(1))
        c = self.fsub.basix_cell()
        Wf = fem.functionspace(self.fsub, basix.ufl.mixed_element([basix.ufl.element("Lagrange", c, 2, shape=(2,)), basix.ufl.element("Lagrange", c, 1)]))
        Vf, _ = Wf.sub(0).collapse()
        if flow == "stokes":
            (u, p), (v, qq) = ufl.TrialFunctions(Wf), ufl.TestFunctions(Wf)
            a = (ufl.inner(ufl.grad(u), ufl.grad(v)) - p * ufl.div(v) - qq * ufl.div(u)) * ufl.dx
            Lf = fem.Constant(self.fsub, 0.0) * qq * ufl.dx
            uin = fem.Function(Vf); uin.interpolate(lambda x: np.vstack([6 * x[1] * (1 - x[1]), 0 * x[1]]))
            inl = mesh.locate_entities_boundary(self.fsub, fdim, lambda x: np.isclose(x[0], 0))
            wl = mesh.locate_entities_boundary(self.fsub, fdim, lambda x: np.isclose(x[1], 0) | np.isclose(x[1], 1))
            bcs = [fem.dirichletbc(uin, fem.locate_dofs_topological((Wf.sub(0), Vf), fdim, inl), Wf.sub(0)),
                   fem.dirichletbc(fem.Function(Vf), fem.locate_dofs_topological((Wf.sub(0), Vf), fdim, wl), Wf.sub(0))]
            wh = LinearProblem(a, Lf, bcs=bcs, petsc_options_prefix=f"st{cnt}_", petsc_options=LU).solve()
            self.uf = wh.sub(0).collapse()
        else:
            self.uf = fem.Function(Vf); self.uf.interpolate(lambda x: np.vstack([6 * x[1] * (1 - x[1]), 0 * x[1]]))
        # transfer to the parent mesh: zero in the solid
        Vp = fem.functionspace(msh, ("Lagrange", 2, (2,))); self.u = fem.Function(Vp)
        nsub = self.fsub.topology.index_map(tdim).size_local + self.fsub.topology.index_map(tdim).num_ghosts
        sub_cells = np.arange(nsub, dtype=np.int32); parent_cells = self.fem_.sub_topology_to_topology(sub_cells, inverse=False)
        self.u.interpolate(self.uf, cells0=sub_cells, cells1=parent_cells)     # cells0: of the SOURCE (submesh), cells1: of the target
        self.u.x.scatter_forward()
        # ── temperature ──
        bot = mesh.locate_entities_boundary(msh, fdim, lambda x: np.isclose(x[1], -self.t))
        out = mesh.locate_entities_boundary(msh, fdim, lambda x: np.isclose(x[0], self.L))
        inl = mesh.locate_entities_boundary(msh, fdim, lambda x: np.isclose(x[0], 0))
        tags = np.concatenate([bot, out, inl]); vals = np.r_[np.full(len(bot), 1), np.full(len(out), 2), np.full(len(inl), 3)].astype(np.int32)
        o = np.argsort(tags); self.ds = ufl.Measure("ds", domain=msh, subdomain_data=mesh.meshtags(msh, fdim, tags[o], vals[o]))
        self.dx = ufl.Measure("dx", domain=msh, subdomain_data=ct)
        self.mode = mode
        if mode == "single":
            S = fem.functionspace(msh, ("Lagrange", deg)); self.T = fem.Function(S); w = ufl.TestFunction(S)
            F = (Pe * ufl.dot(self.u, ufl.grad(self.T)) * w + ufl.dot(ufl.grad(self.T), ufl.grad(w))) * self.dx(1) \
                + ks * ufl.dot(ufl.grad(self.T), ufl.grad(w)) * self.dx(2) - q * w * self.ds(1)
            fin = np.intersect1d(inl, mesh.locate_entities(msh, fdim, lambda x: x[1] >= -1e-12))      # fluid part of the inlet: T = 0
            self.bcs = [fem.dirichletbc(fem.Function(S), fem.locate_dofs_topological(S, fdim, fin))]
            self.F_single = F
            self.pb = NonlinearProblem(F, self.T, bcs=self.bcs, petsc_options_prefix=f"ct{cnt}_", petsc_options=NEWTON)
        else:
            self.ssub, self.sem_, _, _ = mesh.create_submesh(msh, tdim, ct.find(2))
            Sf = fem.functionspace(self.fsub, ("Lagrange", deg)); Ss = fem.functionspace(self.ssub, ("Lagrange", deg))
            self.Tf, self.Ts = fem.Function(Sf), fem.Function(Ss); wf, ws = ufl.TestFunction(Sf), ufl.TestFunction(Ss)
            data = interface_data(msh, ct, 1, 2)
            dS = ufl.Measure("dS", domain=msh, subdomain_data=[(1, data)])(1)
            jump = self.Ts("-") - self.Tf("+")                                   # '+' = fluid
            Ff = (Pe * ufl.dot(self.u, ufl.grad(self.Tf)) * wf + ufl.dot(ufl.grad(self.Tf), ufl.grad(wf))) * self.dx(1) - hc * jump * wf("+") * dS
            Fs = ks * ufl.dot(ufl.grad(self.Ts), ufl.grad(ws)) * self.dx(2) + hc * jump * ws("-") * dS - q * ws * self.ds(1)
            finl = mesh.locate_entities_boundary(self.fsub, fdim, lambda x: np.isclose(x[0], 0))
            self.bcs = [fem.dirichletbc(fem.Function(Sf), fem.locate_dofs_topological(Sf, fdim, finl))]
            self.pb = NonlinearProblem([Ff, Fs], [self.Tf, self.Ts], bcs=self.bcs, entity_maps=[self.fem_, self.sem_], kind="mpi",
                                       petsc_options_prefix=f"cc{cnt}_", petsc_options=dict(NEWTON, snes_rtol=rtol))
            self.T = None

    def solve(self):
        self.pb.solve(); s = self.pb.solver
        return int(s.getConvergedReason()), int(s.getIterationNumber())

    def column(self, x, n=301):
        """y and T(x, y) through solid and fluid"""
        if self.mode == "single":
            return column(self.msh, self.T, x, -self.t, 1.0, n)
        ys, Ts = column(self.ssub, self.Ts, x, -self.t, 0.0, n // 3)
        yf, Tf = column(self.fsub, self.Tf, x, 0.0, 1.0, n)
        return np.r_[ys, yf], np.r_[Ts, Tf]

    def bulk(self, x, n=401):
        """mixing-cup temperature of the fluid at x (mean velocity 1)"""
        msh, T = (self.msh, self.T) if self.mode == "single" else (self.fsub, self.Tf)
        y, Tx = column(msh, T, x, 0.0, 1.0, n); _, ux = column(self.msh, self.u, x, 0.0, 1.0, n)
        return np.trapezoid(ux * Tx, y) / np.trapezoid(ux, y)

    def balance(self):
        """heat balance (per unit depth): heat in through the heated wall = convection out - convection in + conduction out through the ends.
        The end conduction uses the consistent (residual) flux at the Dirichlet inlet and the gradient elsewhere."""
        I = lambda e: comm.allreduce(fem.assemble_scalar(fem.form(e)), op=MPI.SUM)
        if self.mode != "single":
            raise ValueError("balance() is implemented for mode='single'")
        T = self.T; n = ufl.FacetNormal(self.msh); k = ufl.conditional(ufl.gt(ufl.SpatialCoordinate(self.msh)[1], 0), 1.0, self.ks)
        qin = I(self.q * self.ds(1))
        conv = I(self.Pe * self.u[0] * T * self.ds(2)) - I(self.Pe * self.u[0] * T * self.ds(3))
        cond = I(-k * ufl.dot(ufl.grad(T), n) * (self.ds(2) + self.ds(3)))
        # consistent flux at the Dirichlet inlet: the residual (assembled without boundary conditions) summed over the inlet dofs
        import dolfinx.fem.petsc as fp
        from petsc4py import PETSc
        R = fp.assemble_vector(fem.form(self.F_single)); R.ghostUpdate(addv=PETSc.InsertMode.ADD, mode=PETSc.ScatterMode.REVERSE)
        dofs = self.bcs[0].dof_indices()[0]; nown = self.T.function_space.dofmap.index_map.size_local
        react = comm.allreduce(R.array[dofs[dofs < nown]].sum(), op=MPI.SUM); R.destroy()
        return dict(q_in=qin, convected=conv, conducted_ends_gradient=cond, inlet_consistent=-react, residual_gradient=qin - conv - cond, residual_consistent=qin - conv + react)

def interface_data(msh, ct, a=1, b=2):
    """owned interior facets between cells tagged a and b: [cell_a, local facet, cell_b, local facet, ...] ('+' = side a)"""
    tdim = msh.topology.dim; fdim = tdim - 1
    msh.topology.create_connectivity(fdim, tdim); msh.topology.create_connectivity(tdim, fdim)
    f2c, c2f = msh.topology.connectivity(fdim, tdim), msh.topology.connectivity(tdim, fdim)
    tag = np.zeros(msh.topology.index_map(tdim).size_local + msh.topology.index_map(tdim).num_ghosts, dtype=np.int32)
    tag[ct.indices] = ct.values
    data = []
    for f in range(msh.topology.index_map(fdim).size_local):
        cs = f2c.links(f)
        if len(cs) == 2 and {tag[cs[0]], tag[cs[1]]} == {a, b}:
            c0, c1 = (cs[0], cs[1]) if tag[cs[0]] == a else (cs[1], cs[0])
            data += [c0, int(np.where(c2f.links(c0) == f)[0][0]), c1, int(np.where(c2f.links(c1) == f)[0][0])]
    return np.array(data, dtype=np.int32)

def developed_exact(y, t, ks, q=1.0, hc=np.inf):
    """fully developed temperature with uniform flux q (u = 6 y (1 - y), Pe dTb/dx = q), relative to the fluid-side interface value:
    fluid theta(y) = -q y + q (y^3 - y^4/2); solid theta = -q y / ks + q/hc (jump at the interface). Returns theta(y) and theta_bulk."""
    y = np.asarray(y, dtype=float)
    th = np.where(y >= 0, -q * y + q * (y ** 3 - y ** 4 / 2), -q * y / ks + (q / hc if np.isfinite(hc) else 0.0))
    yy = np.linspace(0, 1, 200001); thb = np.trapezoid(6 * yy * (1 - yy) * (-q * yy + q * (yy ** 3 - yy ** 4 / 2)), yy)
    return th, thb

## Q1 — Theory (15 pts)

**(a) (4)** Solve $\mu_e u''-\frac{\mu}{K}u+G=0$, $u(0)=u(1)=0$, and show that $Q=\frac{GK}{\mu}\big(1-2\lambda\tanh\frac1{2\lambda}\big)$, $\lambda=\sqrt{\mu_eK/\mu}$. What are the limits $K\to\infty$ and $K\to0$?

**(b) (4)** In the one-domain approach $1/K$ jumps from 0 (fluid) to $1/K$ (porous) at $y=a$. Which interface conditions does the weak form impose there, and why does it need none written explicitly?

**(c) (4)** Derive the fully developed temperature of the channel heated from one side through a solid wall (lecture §C2) and $Nu_H=35/13$. Why does $k_s$ not appear in $Nu$, and where does a contact conductance $h_c$ appear?

**(d) (3)** Explain why the heat leaving through the Dirichlet inlet should be evaluated as a residual of the weak form and not from $\partial_nT_h$.

> *your answer*

## Q2 — Resolving the Brinkman layer: uniform and graded meshes (14 pts)

$K=10^{-4}$. A graded mesh: map the uniform $s\in[0,1]$ to $y=\frac12(1-\cos\pi s)$ (cells clustered at both walls).

**(a) (8)** For 16, 32, 64 cells: the maximum velocity error and the wall shear stress $\tau_w=u'(0)$ (exact $\frac{GK}{\lambda}\tanh\frac1{2\lambda}$) on uniform and graded meshes.

**(b) (6)** How many cells does each mesh need for a 1 % velocity? What is the size of the first cell of the graded mesh against $\lambda$?

In [ ]:
%%fenicsx -np 2
# TODO: Q2 — uniform vs graded meshes for K = 1e-4

> *your answer (Q2)*

## Q3 — The porous interface: slip, shear and the Beavers–Joseph coefficient (14 pts)

Lower half porous ($K$), upper half free, $G=1$, interface on a mesh line, 128 cells. Beavers and Joseph describe the free flow by a slip condition at the interface: $\partial_yu=\frac{\alpha}{\sqrt K}(u_i-u_D)$ with $u_D=GK/\mu$
the Darcy velocity deep in the layer.

**(a) (8)** For $K=10^{-2},10^{-3},10^{-4}$ compute $u_i=u(a)$, the shear rate on the free side and $\alpha$; compare with the exact two-layer solution.

**(b) (6)** Which $\alpha$ does the Brinkman model imply ($\mu_e=\mu$)? Why is the one-domain model a convenient alternative to imposing the slip — and what remains uncertain in it?

In [ ]:
%%fenicsx -np 2
# TODO: Q3 — interface velocity, shear rate and alpha

> *your answer (Q3)*

## Q4 — Forchheimer: the Ergun pressure drop of a packed bed (13 pts)

Ergun: $\frac{\Delta p}{L}=\frac{150\,\mu(1-\varepsilon)^2}{\varepsilon^3d_p^2}U+\frac{1.75\,\rho(1-\varepsilon)}{\varepsilon^3d_p}U^2$, i.e. Darcy $1/K=\frac{150(1-\varepsilon)^2}{\varepsilon^3d_p^2}$ and Forchheimer $\beta=\frac{1.75\rho(1-\varepsilon)}{\varepsilon^3d_p}$.

**(a) (7)** A bed of $d_p=0.05$ (in units of the channel height), $\varepsilon=0.4$, $\mu=\rho=1$: compute $1/K$, $\beta$; solve Brinkman–Forchheimer for $G=10^5,10^7,10^8,10^9$ (64 cells) and compare the core velocity with Ergun. Newton iterations?

**(b) (6)** Plot (or tabulate) $G/U$ against the particle Reynolds number $Re_p=\rho Ud_p/\mu$. Where does the inertial term double the pressure drop? Is the Brinkman layer resolved at this $K$?

In [ ]:
%%fenicsx -np 2
# TODO: Q4 — Ergun bed

> *your answer (Q4)*

## Q5 — Conjugate heat transfer: the wall conductivity (15 pts)

Lab §5 set-up ($L=40$, $Pe=20$, $q=1$, 160 × 30) with $k_s=0.1,1,10,100$.

**(a) (8)** The heat flux that enters the fluid, $q_i(x)=Pe\,dT_b/dx$, at $x=0.5,2,8,20$, the local $Nu$ (diameter $2H$) at the same points, and the maximum solid temperature.

**(b) (7)** Explain the trend of $q_i$ near the inlet with $k_s$, and why $Nu$ at $x=20$ is the same for all $k_s$ (except a small deviation for $k_s=100$ — why?).

In [ ]:
%%fenicsx -np 2
# TODO: Q5 — wall conductivity

> *your answer (Q5)*

## Q6 — Contact resistance and the conditioning of a penalty (14 pts)

`mode="contact"`: the temperatures of fluid and solid live on two submeshes and exchange $h_c(T_s-T_f)$ on the interface (Week 10's interface integral).

**(a) (7)** For $h_c=10,10^2,10^4,10^6,10^8$: SNES reason and iterations, the jump at $x=20$ against $q/h_c$, and the bulk temperature at $x=20$. Repeat with `rtol=1e-8`.

**(b) (7)** Why does Newton report "diverged" for large $h_c$ although the jump is right to 6 digits? How should one impose perfect contact instead?

In [ ]:
%%fenicsx -np 2
# TODO: Q6 — contact conductance sweep

> *your answer (Q6)*

## Q7 — Debugging an LLM-generated heat-transfer script (15 pts)

An LLM was asked for "a DOLFINx 0.11 script for conjugate heat transfer: laminar channel flow ($Pe=20$, height 1) over a solid wall (thickness 0.5, $k_s=10$) heated from below with flux $q=1$, length 20, that computes the
fully developed Nusselt number and checks the energy balance". It returned the script below.

**(a) (3)** Run it as received (`-np 2`). What does it claim?

**(b) (7)** Find the defects. For each: what it does to the result, whether the script's own check can see it, and a test that exposes it.

**(c) (5)** Correct it (the module may be used) and verify $Nu$ and the energy balance.

In [ ]:
%%fenicsx -np 2
# LLM-generated (as received): conjugate heat transfer — laminar channel flow over a heated solid wall
import numpy as np, ufl
from mpi4py import MPI
from dolfinx import fem, mesh, geometry
from dolfinx.fem.petsc import LinearProblem
comm = MPI.COMM_WORLD
L, t, ks, Pe, q = 20.0, 0.5, 10.0, 20.0, 1.0          # channel height 1, wall thickness t, conductivity ratio, Peclet number, wall heat flux
msh = mesh.create_rectangle(comm, [[0, -t], [L, 1]], [80, 30], mesh.CellType.quadrilateral)
V = fem.functionspace(msh, ("Lagrange", 2))
T, w = ufl.TrialFunction(V), ufl.TestFunction(V)
x, y = ufl.SpatialCoordinate(msh)
u = ufl.as_vector([ufl.conditional(ufl.gt(y, 0), 6 * y * (1 - y), 0.0), 0])   # Poiseuille flow in the fluid, at rest in the solid
k = ufl.conditional(ufl.gt(y, 0), 1.0, ks)                                      # fluid / solid conductivity
a = (Pe * ufl.dot(u, ufl.grad(T)) * w + k * ufl.dot(ufl.grad(T), ufl.grad(w))) * ufl.dx
bottom = mesh.locate_entities_boundary(msh, 1, lambda x: np.isclose(x[1], -t))
ds = ufl.Measure("ds", domain=msh, subdomain_data=mesh.meshtags(msh, 1, np.sort(bottom), np.ones(len(bottom), dtype=np.int32)))
inlet = fem.locate_dofs_geometrical(V, lambda x: np.isclose(x[0], 0.0))       # inlet temperature 0
Th = LinearProblem(a, q * w * ds(1), bcs=[fem.dirichletbc(fem.Constant(msh, 0.0), inlet, V)], petsc_options_prefix="cht_",
                   petsc_options={"ksp_type": "preonly", "pc_type": "lu", "pc_factor_mat_solver_type": "mumps"}).solve()
# fully developed Nusselt number at the outlet: heated-wall temperature and bulk temperature
yy = np.linspace(0, 1, 201); pts = np.zeros((202, 3)); pts[:, 0] = L; pts[:201, 1] = yy; pts[201, 1] = -t
tree = geometry.bb_tree(msh, 2); cc = geometry.compute_colliding_cells(msh, geometry.compute_collisions_points(tree, pts), pts)
vals = np.full(202, -1e300)
for i in range(202):
    if len(cc.links(i)):
        vals[i] = Th.eval(pts[i], cc.links(i)[:1])[0]
comm.Allreduce(MPI.IN_PLACE, vals, op=MPI.MAX)
uu = 6 * yy * (1 - yy); Tb = np.trapezoid(uu * vals[:201], yy) / np.trapezoid(uu, yy); Tw = vals[201]
Nu = 2 * q / (Tw - Tb)                                                          # hydraulic diameter 2H
heat_in, heat_out = q * L, Pe * np.trapezoid(uu * vals[:201], yy)
ok = abs(Nu / 5.385 - 1) < 0.05 and heat_out > 0.9 * heat_in
print(f"Nu_Dh = {Nu:.3f} (one wall heated, the other adiabatic: 5.385); energy in {heat_in:.3f}, carried out {heat_out:.3f} ({100 * heat_out / heat_in:.1f} %)  ->  {'verified' if ok else 'CHECK'}")

In [ ]:
%%fenicsx -np 2
# TODO: Q7(c) — the corrected script and its verification

> *your list of defects and tests (Q7b), and your verification (Q7c)*

## 🤖 AI-usage log

*Required in every assignment and project. Using an LLM is encouraged and never penalized — an empty or vague log is.*

| # | Tool / model | What I asked (key prompt, shortened) | What it got wrong or missed | How I detected it | How I fixed it |
|---|---|---|---|---|---|
| 1 | | | | | |
| 2 | | | | | |
| 3 | | | | | |

**Code provenance** — which cells/functions were mostly generated, which did you write or substantially change?

> …

**Verification** — how do you know the final result is correct (MMS rates, benchmark value, conservation check, parallel = serial)?

> …

**Reflection (2–3 sentences)** — what would you do differently when prompting next time?

> …